# Gemini로 네이버 뉴스 요약 및 분석 챗봇 만들기
[https://ai.google.dev/gemini-api/docs?hl=ko&authuser=1](https://ai.google.dev/gemini-api/docs?hl=ko&authuser=1)

In [ ]:
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-5.4",
    input="Write a short bedtime story about a unicorn."
)

print(response.output_text)

In [3]:
import os
from google import genai
from dotenv import load_dotenv
load_dotenv("./.gemini_env")
api_key = os.getenv("google_api_key")

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="인공지능과 개발자의 미래에 대해 알려줘",
)

print(response.text)

인공지능(AI)의 발전은 개발자의 업무 방식과 역할에 근본적인 변화를 가져오고 있습니다. 많은 이들이 "AI가 개발자를 대체할 것인가?"를 걱정하지만, 대다수의 전문가들은 **"AI를 사용하는 개발자가 AI를 사용하지 않는 개발자를 대체할 것"**이라고 전망합니다.

인공지능과 개발자의 미래를 4가지 핵심 관점에서 정리해 드립니다.

---

### 1. 업무 방식의 변화: '코더(Coder)'에서 '아키텍트(Architect)'로
과거에는 문법을 외우고 코드를 직접 타이핑하는 '구현' 능력이 중요했다면, 미래에는 AI가 그 역할을 대신하게 됩니다.

*   **생산성 폭발:** GitHub Copilot이나 ChatGPT 같은 도구가 보일러플레이트(반복적) 코드 작성, 버그 수정, 테스트 코드 생성을 도와줍니다. 이로 인해 개발 속도가 수 배 이상 빨라집니다.
*   **설계 중심:** 개발자는 "어떻게(How) 코딩할까"보다 **"무엇을(What) 왜(Why) 만들 것인가"**에 더 집중하게 됩니다. 전체적인 시스템 구조(아키텍트), 보안, 확장성, 비즈니스 로직 설계가 핵심 역량이 됩니다.

### 2. 개발 진입 장벽의 변화
AI 덕분에 프로그래밍 언어를 깊게 모르는 사람도 소프트웨어를 만들 수 있는 시대가 오고 있습니다.

*   **노코드/로우코드의 확산:** 단순한 앱이나 웹 서비스는 AI와의 대화만으로 제작이 가능해집니다.
*   **전문성 양극화:** 단순한 기능을 구현하는 '주니어급' 업무는 AI가 상당 부분 대체할 수 있습니다. 반면, 복잡한 문제를 해결하고 AI가 생성한 코드의 오류를 검증하며 책임을 지는 '고숙련 개발자'의 가치는 더욱 높아질 것입니다.

### 3. 개발자에게 요구되는 새로운 역량
미래의 개발자는 단순히 코드를 잘 짜는 것 이상의 능력을 갖춰야 합니다.

*   **AI 리터러시 (AI 활용 능력):** AI 도구를 적재적소에 활용해 결과물을 뽑아내는 '프롬프트 엔지니어링' 능력이 필수적입니다.
*   **코드 리뷰 및

# UI 붙여서 챗봇 만들기

In [1]:
import os
import gradio as gr
from google import genai
from dotenv import load_dotenv
load_dotenv("./.gemini_env")
api_key = os.getenv("google_api_key")

client = genai.Client(api_key=api_key)

# -----------------------------------
# 2) 챗봇 응답 함수
# -----------------------------------
def chat_with_gemini(message, history):
    """
    message: 사용자가 방금 입력한 질문
    history: 이전 대화 내역
    """
    try:
        # 이전 대화 내용을 문자열로 정리
        prompt_parts = []

        for user_msg, bot_msg in history:
            prompt_parts.append(f"사용자: {user_msg}")
            prompt_parts.append(f"AI: {bot_msg}")

        prompt_parts.append(f"사용자: {message}")
        prompt_parts.append("AI:")

        prompt = "\n".join(prompt_parts)

        response = client.models.generate_content(
            model="gemini-3-flash-preview",
            contents=prompt,
        )

        return response.text

    except Exception as e:
        return f"오류가 발생했습니다: {e}"

# -----------------------------------
# 3) Gradio UI
# -----------------------------------
demo = gr.ChatInterface(
    fn=chat_with_gemini,
    title="Gemini Gradio 챗봇",
    description="google-genai와 Gradio를 사용한 간단한 챗봇 예제",
    textbox=gr.Textbox(
        placeholder="메시지를 입력하세요...",
        container=False,
        scale=7
    ),
)

# -----------------------------------
# 4) 실행
# -----------------------------------
demo.launch(inline=False, share=False)

/home/haram/miniforge3/envs/gemini/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/haram/miniforge3/envs/gemini/lib/python3.10/site-packages/gradio/chat_interface.py:334: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


In [2]:
demo.close()

Closing server running on port: 7860


# 네이버에서 뉴스 제목 수집 후 gemini로 내용 요약하기
* 1. 네이버에서 뉴스 수집
* 2. 제목 가져오기
* 3. 제목 선택
* 4. 기사내용 가져오기
* 5. 기사 내용 gemini에게 요약시키기

# 네이버 API에서 뉴스 수집

In [4]:
import os
import re
import requests
import pandas as pd
from dotenv import load_dotenv
load_dotenv("./.gemini_env")
from google import genai
from bs4 import BeautifulSoup as bs



def text_clean(text):
    temp = re.sub(r"</?[^>]+>", "", text)
    temp = re.sub("[^가-힣a-zA-Z09]", " ", temp)
    temp = temp.replace("\n", " ").replace("\t", " ").replace("  ", " ").replace("  ", " ")
    temp = temp.replace("  ", " ").strip()
    return temp

# 네이버 뉴스 검색 함수
def search_news(keyword):
#     keyword = "삼성전자"
    url = "https://openapi.naver.com/v1/search/news"
    payload = dict(query=keyword, display=100, start=1, sort='date')
    headers = {"X-Naver-Client-Id" : os.getenv("Client_Id"), 
               "X-Naver-Client-Secret" : os.getenv("Client_Secret")}
    r = requests.get(url, params=payload, headers=headers)
    print(r.url)
    print(r.status_code)
    data = r.json()
    result = {}
    for item in data['items']:
        for key, value in item.items():
            if key in ('title', 'description'):
                value = text_clean(value)
            result.setdefault(key, []).append(value)
    df = pd.DataFrame(result)
    return df[['title', 'originallink']].head(10)



def content_extract(news_10):
    full_text = ""
    for link in news_10['originallink']:
        try:
            r = requests.get(link)
        except:
            continue
        soup = bs(r.text, "lxml")
        paragraphs = soup.select('[id*="content"], [class*="content"]')
        
        for tags in paragraphs:
            if len(tags.get_text()) > 20:
                full_text += text_clean(tags.get_text())
    return full_text

def summary_gemini(full_text):
    prompt = f"""다음 기사를 주제별로 분류해서 한국어로 주제별 500자 이내로 요약해줘.
                그리고 이 기사 내용이 핀테크 분야와 경제 상황에 미칠 영향을 자세히 분석해 줘:\n\n
                {full_text}
               """
    api_key = os.getenv("google_api_key")

    client = genai.Client(api_key=api_key)

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )

    return response.text


keyword = input()
news_10 = search_news(keyword)
full_text = content_extract(news_10)
print(len(full_text))
final_result = summary_gemini(full_text)
print(final_result)

한국항공우주
https://openapi.naver.com/v1/search/news?query=%ED%95%9C%EA%B5%AD%ED%95%AD%EA%B3%B5%EC%9A%B0%EC%A3%BC&display=100&start=1&sort=date
200
55238
제공된 기사들을 주제별로 분류하고 요약하며, 핀테크 분야 및 경제 상황에 미칠 영향을 분석했습니다.

---

### 기사 주제별 요약 (각 500자 이내)

**1. 국방/항공산업 (KF 전투기 '보라매')**
한국형 전투기 KF '보라매' 양산 1호기가 출고되며 대한민국 독자 전투기 개발 및 생산 시대가 본격적으로 열렸습니다. 2027년까지 40대 생산을 목표로 하며, KAI는 이를 통해 2027년까지 점진적인 실적 확대를 기대하고 있습니다. KF는 공대공 능력은 우수하나 공대지 능력 강화와 차세대 스텔스 전투기(6세대) 개발 등의 과제가 남아있습니다. 전문가들은 국내 수요만으로는 장기적인 성장에 한계가 있어 UAE 등 해외 수출 수주가 반드시 필요하다고 강조하며, 이를 통해 항공우주 강국으로 도약해야 한다고 조언합니다. 경남도는 사천을 항공산업 거점으로 육성할 계획입니다.

**2. 경제/투자**
증권가는 반도체 업종을 실적 기대주로 꼽는 가운데, 중동 긴장 완화 기대감에 금값이 깜짝 반등했습니다. 개인종합자산관리계좌(ISA)는 200만 명이 활용하는 절세 필수템으로 자리매김했으며, ETF 편입 비중이 3년 새 3배로 급증했습니다. 반면 엔터주는 BTS 재결합 기대에도 목표가와 멀어지고 있어 우려를 낳고 있습니다. K-배터리 산업은 전기차(EV) 판매 둔화에 대응해 ESS(에너지저장장치) 시장으로 전환을 모색 중이며, 증권사 리포트에서 하락 종목의 목표가 하향 의견이 부족하다는 지적도 제기되었습니다.

**3. 핀테크/가상자산**
개인종합자산관리계좌(ISA)에서 상장지수펀드(ETF) 편입 비중이 급증하며 젊은 세대부터 은퇴자까지 '절세 고수'들의 필수 투자 도구로 떠올랐습니다. 이는 투자자들이

In [5]:
import os
import re
import requests
import pandas as pd
import gradio as gr

from dotenv import load_dotenv
from google import genai
from bs4 import BeautifulSoup as bs

load_dotenv("./.gemini_env")


# ---------------------------
# 텍스트 정리 함수
# ---------------------------
def text_clean(text):
    temp = re.sub(r"</?[^>]+>", "", str(text))
    temp = re.sub(r"[^가-힣a-zA-Z0-9]", " ", temp)
    temp = re.sub(r"\s+", " ", temp).strip()
    return temp


# ---------------------------
# 네이버 뉴스 검색 함수
# ---------------------------
def search_news(keyword):
    url = "https://openapi.naver.com/v1/search/news"
    payload = {
        "query": keyword,
        "display": 10,   # 우선 10개만
        "start": 1,
        "sort": "date"
    }
    headers = {
        "X-Naver-Client-Id": os.getenv("Client_Id"),
        "X-Naver-Client-Secret": os.getenv("Client_Secret")
    }

    r = requests.get(url, params=payload, headers=headers, timeout=10)
    r.raise_for_status()

    data = r.json()
    items = data.get("items", [])

    result = {}
    for item in items:
        for key, value in item.items():
            if key in ("title", "description"):
                value = text_clean(value)
            result.setdefault(key, []).append(value)

    if not result:
        return pd.DataFrame(columns=["title", "originallink"])

    df = pd.DataFrame(result)

    needed_cols = [col for col in ["title", "originallink"] if col in df.columns]
    return df[needed_cols].head(10)


# ---------------------------
# 뉴스 본문 추출 함수
# ---------------------------
def content_extract(news_df):
    full_text = ""
    collected_links = []

    if news_df is None or news_df.empty:
        return full_text, collected_links

    for link in news_df["originallink"]:
        try:
            r = requests.get(link, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
            r.raise_for_status()
        except Exception:
            continue

        soup = bs(r.text, "lxml")

        # id나 class에 content가 포함된 태그 전부 탐색
        paragraphs = soup.select('[id*="content"], [class*="content"]')

        page_text = []
        for tag in paragraphs:
            txt = text_clean(tag.get_text(" ", strip=True))
            if len(txt) > 20:
                page_text.append(txt)

        if page_text:
            full_text += " ".join(page_text) + "\n"
            collected_links.append(link)

    return full_text.strip(), collected_links


# ---------------------------
# Gemini 요약 함수
# ---------------------------
def summary_gemini(full_text):
    if not full_text.strip():
        return "본문을 추출하지 못했습니다. 다른 키워드로 다시 시도해 주세요."

    prompt = f"""
다음 뉴스 기사들을 주제별로 분류해서 한국어로 정리해줘.

요구사항:
1. 주제별로 묶어라.
2. 각 주제 요약은 500자 이내로 작성해라.
3. 마지막에는 '핀테크 분야 영향'과 '경제 전반 영향'을 구분해서 자세히 분석해라.
4. 전체 답변은 보기 쉽게 제목과 항목을 나눠서 작성해라.

기사 원문:
{full_text}
"""

    api_key = os.getenv("google_api_key")
    client = genai.Client(api_key=api_key)

    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=prompt
    )

    return response.text


# ---------------------------
# 챗봇 응답 함수
# ---------------------------
def chatbot_response(message, history):
    keyword = message.strip()

    if not keyword:
        return "검색할 키워드를 입력해 주세요."

    try:
        news_df = search_news(keyword)

        if news_df.empty:
            return f"'{keyword}'에 대한 뉴스 검색 결과가 없습니다."

        full_text, used_links = content_extract(news_df)

        if not full_text:
            return (
                f"'{keyword}' 뉴스는 찾았지만 본문 추출에 실패했어요.\n\n"
                "가능한 원인:\n"
                "- 언론사 페이지 구조가 달라서 본문 선택이 안 됨\n"
                "- 접속 차단 또는 동적 렌더링 페이지\n"
            )

        summary = summary_gemini(full_text)

        news_list_text = "\n".join(
            [f"{i+1}. {title}" for i, title in enumerate(news_df["title"].tolist())]
        )

        link_text = "\n".join([f"- {link}" for link in used_links[:10]])

        final_answer = f"""검색 키워드: {keyword}

[검색된 뉴스 제목]
{news_list_text}

[요약 및 분석]
{summary}

[본문 추출에 사용된 링크]
{link_text}
"""
        return final_answer

    except Exception as e:
        return f"오류가 발생했습니다: {type(e).__name__}: {e}"


# ---------------------------
# 예시 입력
# ---------------------------
examples = [
    ["삼성전자"],
    ["카카오페이"],
    ["비트코인"],
    ["금리"],
    ["핀테크"]
]


# ---------------------------
# Gradio UI
# ---------------------------
with gr.Blocks(title="뉴스 요약 챗봇") as demo:
    gr.Markdown(
        """
# 뉴스 요약 챗봇
네이버 뉴스 검색 결과를 모아서 기사 본문을 추출한 뒤,
Gemini로 주제별 요약과 핀테크/경제 영향 분석을 제공합니다.
"""
    )

    chatbot = gr.Chatbot(height=500, type="messages")
    msg = gr.Textbox(
        label="검색 키워드 입력",
        placeholder="예: 삼성전자, 비트코인, 금리, 핀테크"
    )
    clear_btn = gr.Button("대화 초기화")

    gr.Examples(
        examples=examples,
        inputs=msg
    )

    def user_submit(user_message, history):
        history = history or []
        history.append({"role": "user", "content": user_message})
        return "", history

    def bot_submit(history):
        user_message = history[-1]["content"]
        bot_message = chatbot_response(user_message, history)
        history.append({"role": "assistant", "content": bot_message})
        return history

    msg.submit(user_submit, [msg, chatbot], [msg, chatbot], queue=False).then(
        bot_submit, chatbot, chatbot
    )

    clear_btn.click(lambda: [], None, chatbot, queue=False)

demo.launch(inline=False, share=False)

* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.
